# External validation — Random Forest (real showfile)

- **Model:** `notebooks/models/rf_plasma_bundle.joblib` (fit on synthetic CSV only).
- **Real data:** Metabolomics Workbench wide table — prefer `notebooks/real/showfile_t.txt`, else `real/showfile_t.txt`.
- **Procedure:** one **sample column** at a time (each column = one subject). True labels come from the **`Factors`** row (`Health Status:non-diabetic` → `control`, `diabetic` → `t2d`). Features are aligned to the trained `feature_names` + imputer via `prepare_real_sample_row`.
- **Trained features:** models saved from the training notebooks expect metabolite headers that match the **synthetic CSV** (RefMet-style names such as `2-Hydroxybutyric acid`, `Glucose`, …). Re-train after changing the synthetic schema.
- Fresh `main.py plasma generate` runs now default to a **real-showfile-aligned 12-metabolite panel** on new output files, which should raise overlap from partial matches toward **12/12**. Use a **new output CSV** or delete the old one to get the new panel.
- **Outputs:** console metrics and `notebooks/models/external_validation_rf_showfile.json`.
- External inference applies the **same saved feature preprocessing** from the training bundle before prediction.


In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

_cwd = Path.cwd().resolve()

_nb_root = None
for cand in (_cwd / "notebooks", _cwd, _cwd.parent / "notebooks", _cwd.parent):
    if (cand / "utils" / "plasma_training_common.py").exists():
        _nb_root = cand
        break
if _nb_root is None:
    _nb_root = _cwd
sys.path.insert(0, str(_nb_root))

from utils.plasma_training_common import (
    MODELS_DIR,
    count_aligned_metabolite_features,
    format_label_for_display,
    load_showfile_labeled_samples,
    prepare_real_sample_row,
    resolve_showfile_path,
)
import utils.plasma_training_common as _ptc

REPO_ROOT = Path(_ptc.__file__).resolve().parent.parent
MODELS_DIR.mkdir(parents=True, exist_ok=True)

BUNDLE_PATH = MODELS_DIR / "rf_plasma_bundle.joblib"
if not BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Train first: missing {BUNDLE_PATH}")

bundle = joblib.load(BUNDLE_PATH)
clf = bundle["model"]
imputer = bundle["imputer"]
feature_preprocessing = bundle.get("feature_preprocessing", "none")
le = bundle["label_encoder"]
feature_names = bundle["feature_names"]

show_path = resolve_showfile_path()
print("REPO_ROOT:", REPO_ROOT)
print("Showfile:", show_path)
print("Features:", len(feature_names))
print("Feature preprocessing:", feature_preprocessing)
print("Feature names:", feature_names)

REPO_ROOT: /Users/andreyolishchuk/vossmoos/simulatexp/omicspd/vossSimpleomics/simulatexp-whitepaper-t2d-basic
Showfile: /Users/andreyolishchuk/vossmoos/simulatexp/omicspd/vossSimpleomics/simulatexp-whitepaper-t2d-basic/real/showfile_t.txt
Features: 12
Feature preprocessing: sample_rank
Feature names: ['2-Hydroxybutyric acid', '3-Hydroxybutyric acid', 'Alanine', 'Glucose', 'Glutamine', 'Glycine', 'Isoleucine', 'Leucine', 'Palmitic acid', 'Phenylalanine', 'Tyrosine', 'Valine']


In [2]:
samples = load_showfile_labeled_samples(show_path)
print("Labeled samples (columns with parseable Factors):", len(samples))

n_model_features = len(feature_names)
aligned_counts: list[int] = []
y_true: list[int] = []
y_pred: list[int] = []
y_proba: list[float] = []
rows_out: list[dict] = []

# One real subject per iteration (one MW table column)
for rec in samples:
    sid = rec["sample_id"]
    lab = rec["label"]
    n_al = count_aligned_metabolite_features(rec["metabolites"], feature_names)
    aligned_counts.append(n_al)
    x = prepare_real_sample_row(
        rec["metabolites"],
        feature_names,
        imputer,
        feature_preprocessing=feature_preprocessing,
    )
    pred_i = int(clf.predict(x)[0])
    true_i = int(le.transform([lab])[0])
    proba = clf.predict_proba(x)[0]
    pos = list(le.classes_).index("t2d") if "t2d" in le.classes_ else 1
    y_true.append(true_i)
    y_pred.append(pred_i)
    y_proba.append(float(proba[pos]))
    pred_label = format_label_for_display(str(le.inverse_transform([pred_i])[0]))
    rows_out.append(
        {
            "sample_id": sid,
            "true_label": format_label_for_display(lab),
            "predicted_label": pred_label,
            "n_aligned_metabolites": n_al,
            "correct": pred_i == true_i,
        }
    )

if aligned_counts:
    print(
        "Metabolites matched to model columns (before imputation): "
        f"min={min(aligned_counts)}, mean={sum(aligned_counts) / len(aligned_counts):.1f}, "
        f"max={max(aligned_counts)} (of {n_model_features} model features)"
    )
    print(
        "If mean match is low, most inputs are training medians → weak real signal; "
        "synthetic–real domain shift also hurts."
    )

y_true_a = np.array(y_true, dtype=int)
y_pred_a = np.array(y_pred, dtype=int)
n = len(y_true_a)
acc = float(accuracy_score(y_true_a, y_pred_a)) if n else 0.0

print("=" * 60)
n_correct = sum(1 for r in rows_out if r["correct"])
print(f"Exact match rate (accuracy): {100.0 * acc:.2f}%  ({n_correct}/{n} correct)")
print("=" * 60)
if n:
    print(
        classification_report(
            y_true_a,
            y_pred_a,
            target_names=[format_label_for_display(c) for c in le.classes_],
            digits=3,
            zero_division=0,
        )
    )
    print("Confusion matrix [rows=true, cols=pred]:")
    print(confusion_matrix(y_true_a, y_pred_a, labels=np.arange(len(le.classes_))))
    if len(np.unique(y_true_a)) > 1:
        try:
            print("ROC-AUC (t2d positive):", roc_auc_score(y_true_a, y_proba))
        except Exception as e:
            print("ROC-AUC skipped:", e)
        pr = np.array(y_proba)
        i_ctrl = int(le.transform(["control"])[0])
        i_t2d = int(le.transform(["t2d"])[0])
        m_c = pr[y_true_a == i_ctrl]
        m_t = pr[y_true_a == i_t2d]
        if len(m_c) and len(m_t):
            print(f"Mean P(t2d) | true control: {m_c.mean():.4f}")
            print(f"Mean P(t2d) | true t2d:     {m_t.mean():.4f}")

wrong = [r for r in rows_out if not r["correct"]]
print("Misclassified:", len(wrong))
for r in wrong[:15]:
    print(r)
if len(wrong) > 15:
    print("...")

Labeled samples (columns with parseable Factors): 56
Metabolites matched to model columns (before imputation): min=11, mean=11.9, max=12 (of 12 model features)
If mean match is low, most inputs are training medians → weak real signal; synthetic–real domain shift also hurts.
Exact match rate (accuracy): 80.36%  (45/56 correct)
              precision    recall  f1-score   support

     Control      0.556     0.417     0.476        12
         T2D      0.851     0.909     0.879        44

    accuracy                          0.804        56
   macro avg      0.703     0.663     0.678        56
weighted avg      0.788     0.804     0.793        56

Confusion matrix [rows=true, cols=pred]:
[[ 5  7]
 [ 4 40]]
ROC-AUC (t2d positive): 0.7263257575757576
Mean P(t2d) | true control: 0.5641
Mean P(t2d) | true t2d:     0.6629
Misclassified: 11
{'sample_id': '090309bsesa100_1', 'true_label': 'T2D', 'predicted_label': 'Control', 'n_aligned_metabolites': 12, 'correct': False}
{'sample_id': '090309b

In [3]:
report_d = (
    classification_report(
        y_true_a,
        y_pred_a,
        target_names=[format_label_for_display(c) for c in le.classes_],
        output_dict=True,
        zero_division=0,
    )
    if n
    else {}
)
cm = (
    confusion_matrix(
        y_true_a, y_pred_a, labels=np.arange(len(le.classes_))
    ).tolist()
    if n
    else []
)
roc = None
if n and len(np.unique(y_true_a)) > 1:
    try:
        roc = float(roc_auc_score(y_true_a, y_proba))
    except Exception:
        roc = None

try:
    _show_rel = str(show_path.relative_to(REPO_ROOT))
except ValueError:
    _show_rel = str(show_path)
payload = {
    "model_type": "sklearn_random_forest",
    "bundle_path": str(BUNDLE_PATH.relative_to(REPO_ROOT)),
    "showfile_path": _show_rel,
    "diagnostics": {
        "n_model_features": n_model_features,
        "aligned_metabolites_min": min(aligned_counts) if aligned_counts else None,
        "aligned_metabolites_mean": round(
            sum(aligned_counts) / len(aligned_counts), 4
        )
        if aligned_counts
        else None,
        "aligned_metabolites_max": max(aligned_counts) if aligned_counts else None,
    },
    "n_samples": n,
    "accuracy": acc,
    "accuracy_percent": round(100.0 * acc, 4),
    "roc_auc_t2d_positive": roc,
    "label_classes": [format_label_for_display(c) for c in le.classes_],
    "classification_report": report_d,
    "confusion_matrix": cm,
    "per_sample": rows_out,
}

out_json = MODELS_DIR / "external_validation_rf_showfile.json"
out_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Wrote", out_json)

Wrote /Users/andreyolishchuk/vossmoos/simulatexp/omicspd/vossSimpleomics/simulatexp-whitepaper-t2d-basic/models/external_validation_rf_showfile.json
